# PGFCL v10 — NB1: Data, Models & Baselines

**Runs:** PyG install · data loading · models · utilities · all 6 baselines (3 seeds each)

**Est. time:** ~3-4 hrs on GPU

**After finish:** Save & Run All → add output as input to NB2

**v10 fixes:** in-round SSL removed · mu_encoder=0.01 · tau=0.7 · lam_warmup=2 · focal_gamma=1.0 · LP disabled

In [ ]:
# ── Cell 1: Pinned PyG install ─────────────────────────────────────────────
import subprocess, sys, importlib

def get_torch_cuda_tag():
    import torch
    tv = torch.__version__.split('+')[0]
    cv = torch.version.cuda
    if cv is None:
        return tv, 'cpu'
    major, minor = cv.split('.')[:2]
    return tv, f'cu{major}{minor}'

torch_ver, cuda_tag = get_torch_cuda_tag()
print(f'PyTorch {torch_ver} | CUDA tag: {cuda_tag}')

WHL_URL  = f'https://data.pyg.org/whl/torch-{torch_ver}+{cuda_tag}.html'
PACKAGES = ['torch-geometric','torch-scatter','torch-sparse',
            'torch-cluster','torch-spline-conv']

subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
                       *PACKAGES, '-f', WHL_URL])

missing = []
for pkg in ['torch_geometric','torch_scatter','torch_sparse',
            'torch_cluster','torch_spline_conv']:
    try:
        importlib.import_module(pkg)
        print(f'  OK  {pkg}')
    except ImportError:
        missing.append(pkg)
if missing:
    raise RuntimeError(f'Failed to install: {missing}')
print('All PyG packages installed.')


In [ ]:
import os, glob, random, copy, time, warnings, hashlib
from dataclasses import dataclass, field, asdict
from typing import List, Optional, Tuple, Dict
from scipy import stats as scipy_stats

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import Adam
from torch.optim.lr_scheduler import CosineAnnealingWarmRestarts
from torch_geometric.data import Data
# v9: GATv2Conv replaces GATConv (query-dependent attention, Brody et al. 2022)
from torch_geometric.nn import GATv2Conv, SAGEConv
from sklearn.metrics import (f1_score, roc_auc_score, accuracy_score,
                              precision_score, recall_score, confusion_matrix,
                              precision_recall_curve)
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
warnings.filterwarnings('ignore')


def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark     = False

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')


@dataclass
class ExperimentConfig:
    # Architecture
    hidden:       int   = 128
    emb_dim:      int   = 64
    heads:        int   = 4
    dropout:      float = 0.4
    head_dropout: float = 0.3

    # Federation
    n_clients:  int   = 4
    seeds:      tuple = (42, 123, 7, 456, 789)  # FIX A2: 5 seeds for adequate statistical power
    test_ratio: float = 0.3

    # FIX A1: feature dimensionality flag
    use_dev_features: bool = True  # True=330-dim (raw+dev), False=165-dim (raw only)

    # Training schedule
    global_rounds:        int   = 100
    head_finetune_rounds: int   = 20
    ssl_pretrain_rounds:  int   = 6    # pre-warm only — NOT run per round
    ssl_epochs:           int   = 20
    sup_epochs:           int   = 25
    lr:                   float = 0.005
    lr_min:               float = 0.0005

    # FedProx
    # FIX: mu_encoder lowered 0.05→0.01 (sweep showed 0.05 is on steep downslope)
    mu_encoder: float = 0.01
    mu_head:    float = 0.0

    # SSL — used ONLY during the pre-warm phase, never in the main training loop
    # FIX: tau raised 0.4→0.7 (sweep best); lam_warmup_rounds lowered 5→2 (sweep best)
    tau:            float = 0.7
    ssl_edge_floor: int   = 15
    feat_drop:      float = 0.3
    edge_drop:      float = 0.2

    # Prototype aggregation
    lam_max:              float = 0.6
    lam_warmup_rounds:    int   = 2
    ema_momentum:         float = 0.85
    raw_blend:            float = 0.2
    use_degree_weighting: bool  = True

    # Ablation flags
    use_ssl:     bool = True   # controls pre-warm phase; in-round SSL removed entirely
    use_protos:  bool = True
    use_fedprox: bool = True

    # v8 contribution flags
    use_contrib_agg:  bool  = True
    contrib_floor:    float = 0.1
    use_grad_guard:   bool  = True
    anomaly_thresh:   float = -0.1
    use_saliency:     bool  = True
    saliency_top_k:   int   = 20
    use_calibration:  bool  = True
    ece_bins:         int   = 15

    # v9 flags
    # FIX: focal_gamma lowered 2.0→1.0 (sweep monotonically decreasing; 1.0 best)
    use_focal_loss:   bool  = True
    focal_gamma:      float = 1.0

    # FIX: use_label_prop disabled (No-LabelProp ablation scores higher than full model)
    # lp_alpha raised to 0.9 (best in sweep) so it is safe to re-enable later.
    use_label_prop:   bool  = False
    lp_alpha:         float = 0.9
    lp_steps:         int   = 1      # also reduced from 2 → 1 for when LP is re-enabled

    # FedPer: local-only head fine-tuning, NO head averaging
    use_fedper:       bool  = True

    cosine_T0:        int   = 50

    # Baselines
    baseline_rounds:       int = 50
    baseline_local_epochs: int = 25
    centralized_epochs:    int = 150


def config_hash(cfg: ExperimentConfig) -> str:
    """Short deterministic hash of config — used to detect checkpoint key collisions."""
    d = {k: v for k, v in asdict(cfg).items() if k != 'seeds'}
    raw = str(sorted(d.items())).encode()
    return hashlib.md5(raw).hexdigest()[:8]


CFG = ExperimentConfig()
print(f'Device : {DEVICE}')
print(f'Seeds  : {CFG.seeds}')
print(f'Config hash: {config_hash(CFG)}')
print(f'v10 fixes: mu_encoder={CFG.mu_encoder}, tau={CFG.tau}, ')
print(f'           lam_warmup={CFG.lam_warmup_rounds}, focal_gamma={CFG.focal_gamma}, ')
print(f'           use_label_prop={CFG.use_label_prop}, lp_alpha={CFG.lp_alpha}')
print(f'           in-round SSL: REMOVED (pre-warm only)')


In [ ]:
import pickle, os, glob

LOCAL_CKPT_DIR = '/kaggle/working/pgfcl_checkpoints'
os.makedirs(LOCAL_CKPT_DIR, exist_ok=True)

INPUT_CKPT_DIRS = []
if os.path.exists('/kaggle/input'):
    for hit in glob.glob('/kaggle/input/**/pgfcl_checkpoints', recursive=True):
        if os.path.isdir(hit) and hit not in INPUT_CKPT_DIRS:
            INPUT_CKPT_DIRS.append(hit)
            print(f'  [ckpt] found: {hit}  ({len(os.listdir(hit))} files)')

if not INPUT_CKPT_DIRS:
    print('  [ckpt] No prior checkpoint dirs found in /kaggle/input.')


def _fname(key):
    return key.replace(' ', '_').replace('/', '_') + '.pkl'


def _find(key):
    p = os.path.join(LOCAL_CKPT_DIR, _fname(key))
    if os.path.exists(p):
        return p
    for d in INPUT_CKPT_DIRS:
        p = os.path.join(d, _fname(key))
        if os.path.exists(p):
            return p
    return None


def save_ckpt(key, obj):
    path = os.path.join(LOCAL_CKPT_DIR, _fname(key))
    with open(path, 'wb') as fh:
        pickle.dump(obj, fh, protocol=4)
    kb = os.path.getsize(path) / 1024
    print(f'  [ckpt] saved  -> {key}  ({kb:.0f} KB)')


def load_ckpt(key):
    p = _find(key)
    if p is not None:
        with open(p, 'rb') as fh:
            obj = pickle.load(fh)
        src = 'local' if LOCAL_CKPT_DIR in p else 'prior-run'
        print(f'  [ckpt] loaded <- {key}  ({src})')
        return obj
    return None


def run_or_load(key, fn, expected_hash=None):
    """Load checkpoint if available, validating config hash to prevent stale-config collisions.

    Pass expected_hash=config_hash(CFG) to reject checkpoints saved under a
    different config (e.g. a prior version with different hyperparameters).
    """
    obj = load_ckpt(key)
    if obj is not None:
        if expected_hash is not None:
            stored_hash = obj.get('config_hash') if isinstance(obj, dict) else None
            if stored_hash is not None and stored_hash != expected_hash:
                print(f'  [ckpt] WARNING: hash mismatch for {key!r} '
                      f'(stored={stored_hash}, expected={expected_hash}) — recomputing')
                obj = None
    if obj is not None:
        return obj
    obj = fn()
    save_ckpt(key, obj)
    return obj


def list_ckpts():
    local = sorted(os.listdir(LOCAL_CKPT_DIR))
    print(f'  Local  ({LOCAL_CKPT_DIR}): {len(local)} file(s)')
    for f in local:
        kb = os.path.getsize(os.path.join(LOCAL_CKPT_DIR, f)) / 1024
        print(f'    {f}  ({kb:.0f} KB)')
    for d in INPUT_CKPT_DIRS:
        files = sorted(os.listdir(d))
        print(f'  Input  ({d}): {len(files)} file(s)')
        for f in files:
            kb = os.path.getsize(os.path.join(d, f)) / 1024
            print(f'    {f}  ({kb:.0f} KB)')


list_ckpts()
print('Checkpoint system ready.')


## Dataset

In [ ]:
ELLIPTIC_DIR = os.environ.get(
    'ELLIPTIC_DIR',
    '/kaggle/input/datasets/organizations/ellipticco/elliptic-data-set/elliptic_bitcoin_dataset'
)
if not os.path.isfile(os.path.join(ELLIPTIC_DIR, 'elliptic_txs_features.csv')):
    hits = glob.glob('/kaggle/input/**/elliptic_txs_features.csv', recursive=True)
    if hits:
        ELLIPTIC_DIR = os.path.dirname(hits[0])
    else:
        for root, _, files in os.walk('/kaggle/input'):
            if 'elliptic_txs_features.csv' in files:
                ELLIPTIC_DIR = root; break
        else:
            raise FileNotFoundError('Cannot find elliptic_txs_features.csv')
print(f'Elliptic dir: {ELLIPTIC_DIR}')


def load_elliptic(data_dir: str = None, use_dev_features: bool = True) -> Data:
    if data_dir is None:
        data_dir = ELLIPTIC_DIR
    print('Loading Elliptic dataset...')
    features = pd.read_csv(f'{data_dir}/elliptic_txs_features.csv', header=None)
    edges    = pd.read_csv(f'{data_dir}/elliptic_txs_edgelist.csv')
    classes  = pd.read_csv(f'{data_dir}/elliptic_txs_classes.csv')

    node_ids  = features.iloc[:, 0].values
    id2idx    = {nid: i for i, nid in enumerate(node_ids)}
    timesteps = features.iloc[:, 1].values.astype(int)
    raw_feats = features.iloc[:, 2:].values.astype(np.float32)

    # FIX: some nodes in the Elliptic CSV have NaN feature values.
    # A single NaN propagates through the per-timestep z-score into the entire
    # timestep block (20 nodes) and then through StandardScaler → inf/NaN inputs
    # to the GNN, causing the "input has nan" error.
    # Replace NaN/inf in raw features with 0 before any further processing.
    raw_feats = np.nan_to_num(raw_feats, nan=0.0, posinf=0.0, neginf=0.0)

    # Per-timestep z-score deviation features (165→330 dims)
    # NOTE for paper: all methods receive 330-dim input; not directly comparable
    # to published results on raw 165-dim features.
    dev_feats = np.zeros_like(raw_feats)
    for t in np.unique(timesteps):
        mask = timesteps == t
        mu   = raw_feats[mask].mean(0)
        sd   = raw_feats[mask].std(0) + 1e-8
        dev_feats[mask] = (raw_feats[mask] - mu) / sd
    # FIX A1: conditional feature concatenation based on use_dev_features flag
    if use_dev_features:
        all_feats = np.concatenate([raw_feats, dev_feats], axis=1)  # 330-dim
    else:
        all_feats = raw_feats.copy()  # 165-dim (for published baseline comparison)
    # Safety clamp after StandardScaler in case any column is still degenerate
    sc = StandardScaler()
    all_feats = sc.fit_transform(all_feats).astype(np.float32)
    all_feats = np.nan_to_num(all_feats, nan=0.0, posinf=0.0, neginf=0.0)

    classes['class'] = classes['class'].map({'1': 1, '2': 0, 'unknown': -1})
    label_map = dict(zip(classes['txId'], classes['class']))
    labels    = np.array([label_map.get(nid, -1) for nid in node_ids])

    valid_edges = [
        (id2idx[u], id2idx[v])
        for u, v in zip(edges.iloc[:, 0], edges.iloc[:, 1])
        if u in id2idx and v in id2idx
    ]
    n_dropped = len(edges) - len(valid_edges)
    if n_dropped:
        print(f'  Warning: dropped {n_dropped} edges')
    srcs, dsts = zip(*valid_edges) if valid_edges else ([], [])
    edge_index = torch.tensor([list(srcs), list(dsts)], dtype=torch.long)

    data = Data(
        x         = torch.tensor(all_feats),
        edge_index = edge_index,
        y          = torch.tensor(labels, dtype=torch.long),
        timestep   = torch.tensor(timesteps, dtype=torch.long)
    )
    print(f'  Nodes: {data.num_nodes:,} | Edges: {data.num_edges:,} | '
          f'Features: {data.num_node_features}')
    print(f'  Illicit: {(labels==1).sum():,} | '
          f'Licit: {(labels==0).sum():,} | '
          f'Unknown: {(labels==-1).sum():,}')
    return data


elliptic_data = load_elliptic(use_dev_features=True)
# FIX A1: also load 165-dim version for published-baseline comparison (C2)
elliptic_data_165 = load_elliptic(use_dev_features=False)
print(f'330-dim data: {elliptic_data.num_node_features} features')
print(f'165-dim data: {elliptic_data_165.num_node_features} features')


## Temporal Federated Split

In [ ]:
# CAVEAT: temporal split produces non-IID clients (different time windows).
# Clients 0 and 3 have ~350-470 illicit examples; clients 1 and 2 have ~1150.
# This is a realistic AML setting but gives the federated model a genuine
# structural disadvantage vs centralised (which sees all time windows together).

def make_mask(n: int, idx: np.ndarray) -> torch.Tensor:
    m = torch.zeros(n, dtype=torch.bool)
    if len(idx):
        m[torch.tensor(np.array(idx), dtype=torch.long)] = True
    return m


def temporal_federated_split(data: Data, cfg: ExperimentConfig):
    labels    = data.y.numpy()
    timesteps = data.timestep.numpy()
    valid_idx  = np.where(labels >= 0)[0]
    sorted_idx = valid_idx[np.argsort(timesteps[valid_idx])]
    splits     = np.array_split(sorted_idx, cfg.n_clients)

    def strat_split(arr):
        if len(arr) == 0:
            return arr, arr
        n_te = max(1, int(len(arr) * cfg.test_ratio))
        return arr[:-n_te], arr[-n_te:]

    clients = []
    for i, split in enumerate(splits):
        illicit        = split[labels[split] == 1]
        licit          = split[labels[split] == 0]
        ill_tr, ill_te = strat_split(illicit)
        lic_tr, lic_te = strat_split(licit)
        tr = np.concatenate([ill_tr, lic_tr])
        te = np.concatenate([ill_te, lic_te])
        clients.append({
            'id':         i,
            'train_mask': make_mask(data.num_nodes, tr),
            'test_mask':  make_mask(data.num_nodes, te),
            'n_train':    len(tr),
            'n_test':     len(te),
        })
        print(f'  Client {i}: train={len(tr):5d} test={len(te):4d} '
              f'ill_train={len(ill_tr):4d} ill_test={len(ill_te):3d}')
    return clients


def get_local_edge_index(data: Data, mask: torch.Tensor,
                          device: torch.device) -> torch.Tensor:
    """Edges where BOTH endpoints are inside mask."""
    ei   = data.edge_index.to(device)
    m    = mask.to(device)
    keep = m[ei[0]] & m[ei[1]]
    return ei[:, keep]


def get_inductive_edge_index(data, train_mask, test_mask, device):
    """Inductive test edges — at least one endpoint in test_mask."""
    ei    = data.edge_index.to(device)
    tr    = train_mask.to(device)
    te    = test_mask.to(device)
    all_m = tr | te
    keep  = (te[ei[1]] & all_m[ei[0]]) | (te[ei[0]] & te[ei[1]])
    return ei[:, keep]


print('=== Elliptic — Temporal Train/Test Split ===')
elliptic_clients = temporal_federated_split(elliptic_data, CFG)


## Models

In [ ]:
class SAGEGATEncoder(nn.Module):
    """
    Encoder: SAGEConv ×2 (skip connections) → GATv2Conv (query-dependent attention).
    forward_with_attention() returns GAT attention weights for saliency logging
    at zero extra compute cost.
    """

    def __init__(self, in_dim: int, cfg: ExperimentConfig):
        super().__init__()
        h, e, dr = cfg.hidden, cfg.emb_dim, cfg.dropout
        self.dropout = dr
        self.conv1 = SAGEConv(in_dim, h)
        self.conv2 = SAGEConv(h, h)
        self.conv3 = GATv2Conv(h, e, heads=1, dropout=dr, concat=False)
        self.skip1 = nn.Linear(in_dim, h, bias=False)
        self.skip2 = nn.Linear(h, h, bias=False)
        self.bn1   = nn.BatchNorm1d(h)
        self.bn2   = nn.BatchNorm1d(h)
        self.proj_head = nn.Sequential(
            nn.Linear(e, e * 2), nn.ReLU(), nn.Linear(e * 2, e)
        )
        self.raw_projector = nn.Linear(in_dim, e, bias=False)

    def _sage_layers(self, x, edge_index):
        h1 = self.conv1(x, edge_index) + self.skip1(x)
        h1 = F.relu(self.bn1(h1))
        h1 = F.dropout(h1, p=self.dropout, training=self.training)
        h2 = self.conv2(h1, edge_index) + self.skip2(h1)
        h2 = F.relu(self.bn2(h2))
        h2 = F.dropout(h2, p=self.dropout, training=self.training)
        return h2

    def forward(self, x, edge_index):
        h2 = self._sage_layers(x, edge_index)
        return self.conv3(h2, edge_index)

    def forward_with_attention(self, x, edge_index):
        h2 = self._sage_layers(x, edge_index)
        emb, (att_edge_index, att_weights) = self.conv3(
            h2, edge_index, return_attention_weights=True
        )
        return emb, att_edge_index, att_weights

    def encode_with_proj(self, x, edge_index):
        z = self.forward(x, edge_index)
        return z, self.proj_head(z)

    def forward_od(self, x, edge_index, width_ratio: float):
        """Ordered Dropout (FjORD) forward pass. Zeroes the tail channels of
        every hidden/output layer beyond width_ratio * layer_width, in a fixed
        channel order, so a low-budget client's active sub-network is always a
        strict nested prefix of every larger client's sub-network. Masked-out
        output channels get exactly zero gradient through the weight rows that
        produce them (verified in isolation), so plain size-weighted FedAvg on
        the full state_dict is a correct aggregator here -- an untouched
        weight row is simply left at the value the client started the round
        with, not corrupted noise.
        """
        def _od_mask(h):
            keep = max(1, round(h.shape[1] * width_ratio))
            mask = torch.zeros_like(h)
            mask[:, :keep] = 1.0
            return h * mask

        h1 = self.conv1(x, edge_index) + self.skip1(x)
        h1 = F.relu(self.bn1(h1))
        h1 = _od_mask(h1)
        h1 = F.dropout(h1, p=self.dropout, training=self.training)
        h2 = self.conv2(h1, edge_index) + self.skip2(h1)
        h2 = F.relu(self.bn2(h2))
        h2 = _od_mask(h2)
        h2 = F.dropout(h2, p=self.dropout, training=self.training)
        emb = self.conv3(h2, edge_index)
        emb = _od_mask(emb)
        return emb


class ClassHead(nn.Module):
    def __init__(self, in_dim: int, cfg: ExperimentConfig, n_classes: int = 2):
        super().__init__()
        h1 = max(128, in_dim * 2)
        h2 = h1 // 2
        self.net = nn.Sequential(
            nn.Linear(in_dim, h1),
            nn.BatchNorm1d(h1),
            nn.ReLU(),
            nn.Dropout(cfg.head_dropout),
            nn.Linear(h1, h2),
            nn.ReLU(),
            nn.Dropout(cfg.head_dropout / 2),
            nn.Linear(h2, n_classes),
        )

    def forward(self, x):
        return self.net(x)


class FullGAT(nn.Module):
    def __init__(self, in_dim: int, cfg: ExperimentConfig):
        super().__init__()
        self.encoder = SAGEGATEncoder(in_dim, cfg)
        self.head    = ClassHead(cfg.emb_dim, cfg)

    def forward(self, x, edge_index, trunc_dim: int = None):
        """trunc_dim: EP-FedProto v3 fixed-tier baseline (BUGFIX). When set,
        zeroes embedding dims >= trunc_dim BEFORE the classification head --
        same post-activation masking principle as encoder.forward_od (FjORD)
        -- so the tier constraint actually restricts what the head can see,
        instead of only touching the auxiliary prototype loss like before.
        None = full emb_dim, unchanged v10 behaviour."""
        emb = self.encoder(x, edge_index)
        if trunc_dim is not None:
            mask = torch.zeros_like(emb)
            mask[:, :trunc_dim] = 1.0
            emb = emb * mask
        return self.head(emb), emb


class SAGEModel(nn.Module):
    """SAGE baseline — same dims as PGFCL for fair comparison."""

    def __init__(self, in_dim: int, cfg: ExperimentConfig, n_classes: int = 2):
        super().__init__()
        h, e = cfg.hidden, cfg.emb_dim
        self.conv1   = SAGEConv(in_dim, h)
        self.conv2   = SAGEConv(h, h)
        self.conv3   = SAGEConv(h, e)
        self.skip1   = nn.Linear(in_dim, h, bias=False)
        self.skip2   = nn.Linear(h, h, bias=False)
        self.bn1     = nn.BatchNorm1d(h)
        self.bn2     = nn.BatchNorm1d(h)
        self.head    = nn.Sequential(
            nn.Linear(e, e * 2), nn.ReLU(), nn.Linear(e * 2, n_classes)
        )
        self.dropout = cfg.dropout

    def forward(self, x, edge_index):
        h1 = F.relu(self.bn1(self.conv1(x, edge_index) + self.skip1(x)))
        h1 = F.dropout(h1, p=self.dropout, training=self.training)
        h2 = F.relu(self.bn2(self.conv2(h1, edge_index) + self.skip2(h1)))
        h2 = F.dropout(h2, p=self.dropout, training=self.training)
        emb = self.conv3(h2, edge_index)
        return self.head(emb), emb

GATEncoder = SAGEGATEncoder
print('Models defined.')


## Utilities

In [ ]:
# ── Evaluation ───────────────────────────────────────────────────────────────

def evaluate_tuned(model, data, train_mask, test_mask, device, cfg=None,
                    proto_dim: int = None):
    """
    Threshold tuned on train set (no leakage). Test inference is inductive.
    Label propagation is applied only if cfg.use_label_prop is True.
    FIX: LP disabled by default (use_label_prop=False) — ablation showed it
    hurts at any reasonable alpha on this graph topology.

    proto_dim: EP-FedProto v3 fixed-tier baseline (BUGFIX). Forwarded as
    trunc_dim into the model so evaluation actually sees the same truncated
    embedding the tier is supposed to constrain, instead of always scoring
    the full-capacity model regardless of tier_d. None = unchanged v10
    behaviour.
    """
    if cfg is None:
        cfg = CFG
    model.eval()

    ei_tr = get_local_edge_index(data, train_mask, device)
    with torch.no_grad():
        logits_tr, _ = model(data.x.to(device), ei_tr, trunc_dim=proto_dim)
    probs_tr = F.softmax(logits_tr, dim=1)[:, 1].cpu().numpy()
    # Guard: NaN in probs means the model has diverged (BN with tiny batch, etc.)
    if not np.isfinite(probs_tr).all():
        return {'acc': 0., 'f1': 0., 'auc': 0., 'prec': 0., 'rec': 0.,
                'cm': np.zeros((2, 2), int), 'thresh': 0.5}

    tr_lab = train_mask & (data.y >= 0)
    best_thresh = 0.5
    if tr_lab.sum() > 0 and len(np.unique(data.y[tr_lab].numpy())) > 1:
        p, r, thresholds = precision_recall_curve(
            data.y[tr_lab].numpy(), probs_tr[tr_lab.numpy()])
        f1s = 2 * p * r / (p + r + 1e-8)
        best_thresh = float(np.clip(thresholds[np.argmax(f1s[:-1])], 0.1, 0.9))

    ei_te = get_inductive_edge_index(data, train_mask, test_mask, device)
    with torch.no_grad():
        logits_te, _ = model(data.x.to(device), ei_te, trunc_dim=proto_dim)
    probs_te_full = F.softmax(logits_te, dim=1)[:, 1]

    if cfg.use_label_prop:
        probs_te_full = label_propagation(
            probs_te_full, ei_te, data.num_nodes,
            alpha=cfg.lp_alpha, steps=cfg.lp_steps
        ).to(device)
    probs_te = probs_te_full.cpu().numpy()

    te_lab = test_mask & (data.y >= 0)
    if te_lab.sum() == 0:
        return {'acc': 0., 'f1': 0., 'auc': 0., 'prec': 0., 'rec': 0.,
                'cm': np.zeros((2, 2), int), 'thresh': best_thresh}

    probs_te_masked = probs_te[te_lab.numpy()]
    preds_te        = (probs_te_masked >= best_thresh).astype(int)
    true_te         = data.y[te_lab].numpy()

    return {
        'acc':    accuracy_score(true_te, preds_te),
        'f1':     f1_score(true_te, preds_te, zero_division=0),
        'auc':    roc_auc_score(true_te, probs_te_masked) if len(np.unique(true_te)) > 1 else 0.,
        'prec':   precision_score(true_te, preds_te, zero_division=0),
        'rec':    recall_score(true_te, preds_te, zero_division=0),
        'cm':     confusion_matrix(true_te, preds_te),
        'thresh': best_thresh
    }


# ── Loss Functions ────────────────────────────────────────────────────────────

def weighted_ce(logits, labels, device):
    n_classes = logits.shape[1]
    counts    = torch.bincount(labels, minlength=n_classes).float().clamp(min=1.0)
    weight    = labels.shape[0] / (n_classes * counts)
    weight    = weight.clamp(0.1, 10.0)
    return F.cross_entropy(logits, labels, weight=weight.to(device))


def focal_loss(logits, labels, device, gamma=1.0):
    """
    Focal loss (Lin et al. 2017) with inverse-frequency class weights.
    FIX: default gamma lowered 2.0→1.0. Sweep showed monotonic decrease
    in F1 as gamma increases; gamma=1.0 is best. gamma=2.0 over-suppresses
    easy examples on this class-imbalanced, non-IID federated setting.
    """
    n_classes = logits.shape[1]
    counts    = torch.bincount(labels, minlength=n_classes).float().clamp(1.0)
    alpha_cls = (labels.shape[0] / (n_classes * counts)).clamp(0.1, 10.0)
    alpha_t   = alpha_cls[labels]
    ce  = F.cross_entropy(logits, labels, reduction='none')
    pt  = torch.exp(-ce).clamp(1e-7, 1.0 - 1e-7)  # clamp prevents (1-pt)^gamma=0 → NaN grad
    fl  = alpha_t * (1.0 - pt) ** gamma * ce
    return fl.mean()


def supervised_loss(logits, labels, device, cfg):
    if cfg.use_focal_loss:
        return focal_loss(logits, labels, device, gamma=cfg.focal_gamma)
    return weighted_ce(logits, labels, device)


def label_propagation(probs, edge_index, n_nodes, alpha=0.9, steps=1):
    """
    Post-processing LP. Disabled by default (use_label_prop=False).
    FIX: alpha default raised 0.8→0.9, steps lowered 2→1 for when it
    is re-enabled. The sweep showed LP becomes harmful below alpha~0.9
    on the inductive test subgraph due to unlabeled-region noise.
    """
    dev = probs.device
    ei  = edge_index.to(dev)
    src_idx, dst_idx = ei[0], ei[1]
    p   = probs.clone()
    p0  = probs.clone()
    deg = torch.zeros(n_nodes, device=dev)
    deg.scatter_add_(0, dst_idx, torch.ones(dst_idx.shape[0], device=dev))
    deg = deg.clamp(min=1.0)
    for _ in range(steps):
        agg = torch.zeros(n_nodes, device=dev)
        agg.scatter_add_(0, dst_idx, p[src_idx])
        agg = agg / deg
        p = alpha * p0 + (1.0 - alpha) * agg
    return p


def infonce_loss(z_proj, edge_index, local_mask, device, cfg: ExperimentConfig):
    z_norm    = F.normalize(z_proj, dim=1)
    feat_mask = torch.rand(z_proj.shape[1], device=device) > cfg.feat_drop
    z_aug     = F.normalize(z_norm * feat_mask.float(), dim=1)
    src, dst = edge_index
    keep     = torch.rand(src.shape[0], device=device) > cfg.edge_drop
    src_k, dst_k = src[keep], dst[keep]
    if src_k.numel() == 0:
        return torch.tensor(0.0, device=device)
    pos_sim = (z_norm[src_k] * z_aug[dst_k]).sum(1) / cfg.tau
    anchor_nodes = torch.cat([src_k, dst_k]).unique()
    local_nodes  = torch.where(local_mask.to(device))[0]
    neg_pool     = local_nodes[~torch.isin(local_nodes, anchor_nodes)]
    if neg_pool.numel() < 4:
        perm     = torch.randperm(src_k.shape[0], device=device)
        neg_pool = src_k[perm]
    n_neg   = min(256, neg_pool.shape[0])
    neg_idx = neg_pool[torch.randperm(neg_pool.shape[0], device=device)[:n_neg]]
    neg_sim = torch.mm(z_norm[src_k], z_norm[neg_idx].T) / cfg.tau
    logits = torch.cat([pos_sim.unsqueeze(1), neg_sim], dim=1)
    target = torch.zeros(logits.shape[0], dtype=torch.long, device=device)
    return F.cross_entropy(logits, target)


def label_aware_edge_supcon(z_proj, data, local_mask, device, cfg):
    """Label-aware SupCon: illicit anchors, illicit positives, licit negatives."""
    labels = data.y.to(device)
    lm = local_mask.to(device)
    ill_mask = lm & (labels == 1)
    lic_mask = lm & (labels == 0)
    if ill_mask.sum() < 2 or lic_mask.sum() < 1:
        return torch.tensor(0.0, device=device)
    z_norm = F.normalize(z_proj, dim=1)
    feat_mask = torch.rand(z_proj.shape[1], device=device) > cfg.feat_drop
    z_aug = F.normalize(z_norm * feat_mask.float(), dim=1)
    ill_idx = torch.where(ill_mask)[0]
    lic_idx = torch.where(lic_mask)[0]
    losses = []
    n_anchors = min(64, ill_idx.shape[0])
    perm = torch.randperm(ill_idx.shape[0], device=device)[:n_anchors]
    anchors = ill_idx[perm]
    for a in anchors:
        pos_pool = ill_idx[ill_idx != a]
        if pos_pool.shape[0] == 0:
            continue
        n_pos = min(8, pos_pool.shape[0])
        pos = pos_pool[torch.randperm(pos_pool.shape[0], device=device)[:n_pos]]
        n_neg = min(32, lic_idx.shape[0])
        neg = lic_idx[torch.randperm(lic_idx.shape[0], device=device)[:n_neg]]
        pos_sim = (z_norm[a] * z_aug[pos]).sum(1) / cfg.tau
        neg_sim = (z_norm[a] * z_aug[neg]).sum(1) / cfg.tau
        logits = torch.cat([pos_sim, neg_sim])
        target = torch.zeros(logits.shape[0], device=device)
        target[:pos_sim.shape[0]] = 1.0 / pos_sim.shape[0]
        log_probs = F.log_softmax(logits, dim=0)
        loss = -(target * log_probs).sum()
        losses.append(loss)
    return torch.stack(losses).mean() if losses else torch.tensor(0.0, device=device)


def prototype_supcon_loss(z, labels, mask, global_protos, device,
                           cfg: ExperimentConfig) -> torch.Tensor:
    labeled = mask.to(device) & (labels.to(device) >= 0)
    if not labeled.any() or global_protos is None:
        return torch.tensor(0.0, device=device)
    z_lab = F.normalize(z[labeled], dim=1)
    y_lab = labels.to(device)[labeled]
    p0 = F.normalize(global_protos[0].detach().unsqueeze(0), dim=1)
    p1 = F.normalize(global_protos[1].detach().unsqueeze(0), dim=1)
    protos_cat   = torch.cat([p0, p1], dim=0)
    proto_logits = torch.mm(z_lab, protos_cat.T) / cfg.tau
    return F.cross_entropy(proto_logits, y_lab)


# ── Aggregation Helpers ────────────────────────────────────────────────────────

def avg_metrics(metric_list):
    keys = ['f1', 'auc', 'prec', 'rec', 'acc']
    out  = {}
    for k in keys:
        vals = [m[k] for m in metric_list if k in m]
        out[k]           = float(np.mean(vals))
        out[k + '_std']  = float(np.std(vals))
        out[k + '_vals'] = vals
    return out


def fedavg_state_selective(global_model, local_models, client_sizes,
                            exclude_keys=None):
    if exclude_keys is None:
        exclude_keys = ['running_mean', 'running_var', 'num_batches_tracked']
    total   = sum(client_sizes)
    g_state = global_model.state_dict()
    new_state = {}
    for key in g_state:
        if any(ex in key for ex in exclude_keys):
            new_state[key] = g_state[key]
            continue
        acc = torch.zeros_like(g_state[key].float())
        for i, lm in enumerate(local_models):
            acc += lm.state_dict()[key].float() * (client_sizes[i] / total)
        new_state[key] = acc.to(g_state[key].dtype)
    global_model.load_state_dict(new_state)
    return global_model


def fedavg_encoder_only(global_model, local_encoders, client_sizes):
    exclude = ['running_mean', 'running_var', 'num_batches_tracked', 'proj_head']
    total   = sum(client_sizes)
    enc_ref = global_model.encoder.state_dict()
    new_enc = {}
    for key in enc_ref:
        if any(ex in key for ex in exclude):
            new_enc[key] = enc_ref[key]
            continue
        acc = torch.zeros_like(enc_ref[key].float())
        for i, enc in enumerate(local_encoders):
            acc += enc.state_dict()[key].float() * (client_sizes[i] / total)
        new_enc[key] = acc.to(enc_ref[key].dtype)
    global_model.encoder.load_state_dict(new_enc)
    return global_model


def compute_client_contrib_weights(client_emb_protos, client_sizes,
                                    global_protos, cfg):
    if global_protos is None:
        total = sum(client_sizes)
        return [s / total for s in client_sizes]
    global_ill = F.normalize(global_protos[1].unsqueeze(0), dim=1).squeeze(0)
    weights = []
    for protos, sz in zip(client_emb_protos, client_sizes):
        local_ill = F.normalize(protos[1].unsqueeze(0), dim=1).squeeze(0)
        cos_sim   = torch.dot(local_ill.float(), global_ill.float()).item()
        quality   = (cos_sim + 1.0) / 2.0
        quality   = cfg.contrib_floor + (1.0 - cfg.contrib_floor) * quality
        weights.append((sz ** 0.5) * quality)
    total = sum(weights)
    return [w / total for w in weights]


def fedavg_state_contrib(global_model, local_models, contrib_weights,
                          exclude_keys=None):
    if exclude_keys is None:
        exclude_keys = ['running_mean', 'running_var', 'num_batches_tracked']
    # Guard: if a client's model has NaN weights (diverged training),
    # exclude it and renormalise the remaining weights.
    nan_clients = []
    for i, lm in enumerate(local_models):
        params = [p for k, p in lm.state_dict().items() if not any(ex in k for ex in exclude_keys)]
        if any(torch.isnan(p.float()).any() for p in params):
            nan_clients.append(i)
    if nan_clients:
        print(f'  [fedavg] WARNING: clients {nan_clients} have NaN weights — excluded from aggregation')
        valid_w = [w if i not in nan_clients else 0.0 for i, w in enumerate(contrib_weights)]
        total_w = sum(valid_w)
        contrib_weights = [w / total_w if total_w > 0 else 1.0 / len(local_models) for w in valid_w]

    g_state   = global_model.state_dict()
    new_state = {}
    for key in g_state:
        if any(ex in key for ex in exclude_keys):
            new_state[key] = g_state[key]
            continue
        acc = torch.zeros_like(g_state[key].float())
        for i, lm in enumerate(local_models):
            if i in nan_clients:
                continue
            acc += lm.state_dict()[key].float() * contrib_weights[i]
        new_state[key] = acc.to(g_state[key].dtype)
    global_model.load_state_dict(new_state)
    return global_model


def compute_update_directions(global_model, local_models):
    exclude = ['running_mean', 'running_var', 'num_batches_tracked']
    g_state = global_model.state_dict()
    directions = []
    for lm in local_models:
        diff_parts = []
        for key in g_state:
            if any(ex in key for ex in exclude):
                continue
            diff = (lm.state_dict()[key].float() - g_state[key].float()).flatten()
            diff_parts.append(diff)
        vec = torch.cat(diff_parts)
        norm = vec.norm()
        directions.append(vec / (norm + 1e-8))
    return directions


def grad_guard_weights(global_model, local_models, client_sizes, cfg):
    directions  = compute_update_directions(global_model, local_models)
    total_size  = sum(client_sizes)
    size_w      = [s / total_size for s in client_sizes]
    consensus = sum(d * w for d, w in zip(directions, size_w))
    consensus = consensus / (consensus.norm() + 1e-8)
    weights, flagged = [], []
    for i, d in enumerate(directions):
        cos_sim = torch.dot(d, consensus).item()
        if cos_sim < cfg.anomaly_thresh:
            weights.append(cfg.contrib_floor)
            flagged.append(i)
        else:
            weights.append(size_w[i])
    total = sum(weights)
    weights = [w / total for w in weights]
    if flagged:
        print(f'  [GradGuard] flagged clients {flagged}')
    return weights, flagged


def extract_node_saliency(model, data, test_mask, device, cfg, top_k=None):
    if top_k is None:
        top_k = cfg.saliency_top_k
    model.eval()
    ei = get_local_edge_index(data, test_mask, device)
    x  = data.x.to(device)
    with torch.no_grad():
        emb, att_ei, att_w = model.encoder.forward_with_attention(x, ei)
        logits = model.head(emb)
        probs  = F.softmax(logits, dim=1)[:, 1].cpu().numpy()
    att_w_flat = att_w.squeeze(-1).cpu()
    dst_nodes = att_ei[1].cpu()
    n_nodes   = data.num_nodes
    node_att  = torch.zeros(n_nodes)
    node_cnt  = torch.zeros(n_nodes)
    for j in range(dst_nodes.shape[0]):
        node_att[dst_nodes[j]] += att_w_flat[j].item()
        node_cnt[dst_nodes[j]] += 1
    node_saliency = node_att / (node_cnt + 1e-8)
    te_lab = test_mask & (data.y >= 0)
    te_idx = torch.where(te_lab)[0].cpu().numpy()
    saliency_te = node_saliency[te_idx].numpy()
    probs_te    = probs[te_idx]
    labels_te   = data.y[te_idx].numpy()
    risk_score  = probs_te * saliency_te
    top_idx     = np.argsort(risk_score)[::-1][:top_k]
    return {
        'node_indices': te_idx[top_idx],
        'risk_scores':  risk_score[top_idx],
        'probs':        probs_te[top_idx],
        'saliency':     saliency_te[top_idx],
        'true_labels':  labels_te[top_idx],
    }


def compute_ece(model, data, test_mask, device, cfg, train_mask=None,
                 proto_dim: int = None):
    """Compute Expected Calibration Error.

    Uses inductive edge index (same as evaluate_tuned) when train_mask is
    provided, ensuring ECE is measured under the same message-passing conditions
    as classification metrics. Falls back to transductive test subgraph if
    train_mask is not supplied.

    proto_dim: EP-FedProto v3 fixed-tier baseline (BUGFIX, same as
    evaluate_tuned) -- forwarded as trunc_dim so calibration is measured on
    the same truncated model as the F1 numbers, not the full-capacity one.
    """
    model.eval()
    if train_mask is not None:
        ei = get_inductive_edge_index(data, train_mask, test_mask, device)
    else:
        ei = get_local_edge_index(data, test_mask, device)
    with torch.no_grad():
        logits, _ = model(data.x.to(device), ei, trunc_dim=proto_dim)
    probs   = F.softmax(logits, dim=1)[:, 1].cpu().numpy()
    te_lab  = test_mask & (data.y >= 0)
    y_true  = data.y[te_lab].numpy()
    y_prob  = probs[te_lab.numpy()]
    n_bins  = cfg.ece_bins
    bins    = np.linspace(0.0, 1.0, n_bins + 1)
    ece     = 0.0
    bin_data = []
    for i in range(n_bins):
        lo, hi = bins[i], bins[i+1]
        mask = (y_prob >= lo) & (y_prob < hi)
        if mask.sum() == 0:
            bin_data.append({'conf': (lo+hi)/2, 'acc': 0., 'count': 0})
            continue
        conf = y_prob[mask].mean()
        acc  = y_true[mask].mean()
        frac = mask.sum() / len(y_true)
        ece += frac * abs(acc - conf)
        bin_data.append({'conf': conf, 'acc': acc, 'count': int(mask.sum())})
    return float(ece), bin_data


print('Utility functions loaded (v10 fixes: LP disabled, focal_gamma=1.0).')


## Training Functions

In [ ]:
# ── SSL pre-training (pre-warm phase ONLY) ────────────────────────────────────

def ssl_pretrain_client(encoder, data: Data, client: dict,
                         device: torch.device, cfg: ExperimentConfig,
                         opt: Optional[torch.optim.Optimizer] = None):
    """
    Label-aware SupCon on one client encoder.
    Full-graph message-passing; contrastive loss restricted to local labeled nodes.
    Called ONLY during ssl_pretrain_phase, never inside the main training loop.
    """
    encoder.train()
    if opt is None:
        opt = Adam(encoder.parameters(), lr=cfg.lr)
    x       = data.x.to(device)
    lm      = client['train_mask'].to(device)
    ei_full = data.edge_index.to(device)
    for _ in range(cfg.ssl_epochs):
        opt.zero_grad()
        _, z_proj = encoder.encode_with_proj(x, ei_full)
        loss = label_aware_edge_supcon(z_proj, data, lm, device, cfg)
        loss.backward()
        opt.step()


def ssl_pretrain_phase(global_model: FullGAT, data: Data, clients: list,
                        device: torch.device, cfg: ExperimentConfig,
                        verbose: bool = True) -> FullGAT:
    """
    FIX: SSL runs ONLY here (pre-warm), never inside the per-round loop.
    Diagnostic A showed in-round SSL (use_ssl=True during main loop) was
    net-negative: No-SSL variant scored F1=0.8125 vs full model F1=0.7655.
    The pre-warm phase alone (Diag A2) recovers all SSL benefit.
    """
    if not cfg.use_ssl or cfg.ssl_pretrain_rounds == 0:
        return global_model
    if verbose:
        print(f'  [SSL pre-warm] {cfg.ssl_pretrain_rounds} rounds (no in-round SSL after this)...')
    sizes = [c['n_train'] for c in clients]
    # Persist only the optimizer *state buffers* (step counts, moments) across
    # rounds, keyed by parameter position rather than tensor identity.
    # Storing full state_dicts over global_model.encoder params is wrong because
    # each round creates a fresh enc with new tensor identities — load_state_dict
    # would silently apply the wrong state. We instead carry raw state dicts and
    # transplant them by position after the new optimizer is created.
    client_ssl_opt_states: list = [None] * len(clients)
    for pre_rnd in range(cfg.ssl_pretrain_rounds):
        local_encoders = []
        for j, client in enumerate(clients):
            enc = SAGEGATEncoder(data.num_node_features, cfg).to(device)
            enc.load_state_dict(global_model.encoder.state_dict())
            opt = Adam(enc.parameters(), lr=cfg.lr)
            # Transplant saved state buffers by parameter position
            if client_ssl_opt_states[j] is not None:
                saved = client_ssl_opt_states[j]
                new_state = opt.state_dict()
                # Remap saved state from old param ids → new param ids
                old_id_to_pos = {pid: pos for pos, pid in enumerate(saved['state'])}
                for new_pid, pg in enumerate(new_state['param_groups'][0]['params']):
                    if new_pid in saved['state']:
                        new_state['state'][new_pid] = saved['state'][new_pid]
                try:
                    opt.load_state_dict(new_state)
                except (ValueError, KeyError):
                    pass  # mismatch on first round — start fresh, not a problem
            ssl_pretrain_client(enc, data, client, device, cfg, opt=opt)
            client_ssl_opt_states[j] = opt.state_dict()
            local_encoders.append(enc)
        global_model = fedavg_encoder_only(global_model, local_encoders, sizes)
        if verbose:
            print(f'    Pre-warm {pre_rnd+1}/{cfg.ssl_pretrain_rounds} done')
    return global_model


# ── Prototype computation ─────────────────────────────────────────────────────

def compute_embedding_prototypes(embeddings, data, mask, device, cfg):
    labels  = data.y.to(device)
    labeled = mask.to(device) & (labels >= 0)
    if cfg.use_degree_weighting:
        ei  = data.edge_index.to(device)
        deg = torch.zeros(data.num_nodes, dtype=torch.float, device=device)
        deg.scatter_add_(0, ei[0], torch.ones(ei.size(1), device=device))
        deg = deg + 1.0
    protos, counts = {}, {}
    for c in [0, 1]:
        cm = labeled & (labels == c)
        counts[c] = int(cm.sum())
        if counts[c] == 0:
            protos[c] = torch.zeros(embeddings.shape[1], device=device)
            continue
        emb_c = embeddings[cm]
        if cfg.use_degree_weighting:
            w = deg[cm].unsqueeze(1)
            protos[c] = (emb_c * w).sum(0) / w.sum()
        else:
            protos[c] = emb_c.mean(0)
    return protos, counts


def chain_prototype_inheritance(client_emb_protos, client_cls_counts,
                                  clients, rnd, cfg):
    if rnd == 0 or client_emb_protos is None:
        return None
    n = len(client_emb_protos)
    client_protos = {}
    for i in range(n):
        if i == 0 or client_emb_protos[i-1] is None:
            own = client_emb_protos[i]
            client_protos[i] = {
                c: F.normalize(own[c].unsqueeze(0), dim=1).squeeze(0)
                for c in [0, 1]
            }
        else:
            pred = client_emb_protos[i-1]
            own  = client_emb_protos[i]
            blended = {}
            for c in [0, 1]:
                p = F.normalize(pred[c].unsqueeze(0), dim=1).squeeze(0)
                o = F.normalize(own[c].unsqueeze(0), dim=1).squeeze(0)
                alpha = cfg.ema_momentum if c == 1 else 0.3
                blended[c] = F.normalize(
                    (alpha * p + (1 - alpha) * o).unsqueeze(0), dim=1
                ).squeeze(0)
            client_protos[i] = blended
    return client_protos


def supervised_round_client(model, data, client, device, cfg,
                              global_model=None, global_protos=None, lam=0.0,
                              proto_dim=None):
    model.train()
    opt   = Adam(model.parameters(), lr=cfg.lr)
    sched = CosineAnnealingWarmRestarts(opt, T_0=max(1, cfg.sup_epochs // 2), eta_min=cfg.lr_min)
    ei    = get_local_edge_index(data, client['train_mask'], device)
    vmask = client['train_mask'] & (data.y >= 0)
    lbls  = data.y[vmask].to(device)
    enc_ref = (
        {n: p.detach().clone() for n, p in global_model.encoder.named_parameters()}
        if global_model is not None and cfg.use_fedprox else None
    )
    # Guard: BatchNorm1d in train mode with a single sample produces NaN weights.
    # This can happen on small client windows in the sensitivity sweep.
    if vmask.sum() < 2:
        return  # skip training — global weights unchanged for this client
    for _ in range(cfg.sup_epochs):
        opt.zero_grad()
        # BUGFIX: forward the same trunc_dim used by the auxiliary prototype
        # loss below, so the classification head itself is capacity-limited
        # by the tier -- not just the extra prototype regularization term.
        logits, z = model(data.x.to(device), ei, trunc_dim=proto_dim)
        loss = supervised_loss(logits[vmask.to(device)], lbls, device, cfg)
        if cfg.use_fedprox and enc_ref is not None and cfg.mu_encoder > 0:
            enc_prox = sum(
                ((p - enc_ref[n]) ** 2).sum()
                for n, p in model.encoder.named_parameters()
            )
            loss = loss + (cfg.mu_encoder / 2) * enc_prox
        if cfg.use_protos and global_protos is not None and lam > 0:
            # EP-FedProto v3: optional fixed-dim truncation of the transmitted
            # prototype (used by the fixed-budget-per-tier baseline). None =
            # full-dim behaviour, unchanged from v10.
            if proto_dim is not None:
                _protos = {c: v[:proto_dim] for c, v in global_protos.items()}
                _z = z[:, :proto_dim]
            else:
                _protos, _z = global_protos, z
            proto_loss = prototype_supcon_loss(
                _z, data.y, client['train_mask'], _protos, device, cfg
            )
            loss = loss + lam * proto_loss
        loss.backward()
        opt.step()
        sched.step()


def fedper_head_finetune(global_model, data, clients, device, cfg, verbose=True,
                          proto_dim: int = None):
    """
    FedPer: each client fine-tunes its own head locally, encoder frozen.
    FIX: heads are NOT averaged back (v9 bug). Each client keeps its own head.
    Returns (client_results, aggregated_metrics).

    proto_dim: EP-FedProto v3 fixed-tier baseline (BUGFIX). Without this,
    FedPer's head fine-tuning and evaluation silently ran at full emb_dim
    regardless of the tier, undoing the truncation applied everywhere else
    in run_pgfcl -- since use_fedper=True by default, this phase runs for
    every fixed-tier seed and would have overwritten the correctly-truncated
    Phase-2 metrics with full-capacity ones whenever FedPer scored higher.
    """
    if not cfg.use_fedper or cfg.head_finetune_rounds == 0:
        all_train = torch.zeros(data.num_nodes, dtype=torch.bool)
        all_test  = torch.zeros(data.num_nodes, dtype=torch.bool)
        for c in clients:
            all_train |= c['train_mask']
            all_test  |= c['test_mask']
        m = evaluate_tuned(global_model, data, all_train, all_test, device, cfg,
                            proto_dim=proto_dim)
        return [(global_model, m)], m

    if verbose:
        print(f'  [FedPer] Local head FT ({cfg.head_finetune_rounds} rounds, encoder frozen)...')

    client_results = []
    for i, client in enumerate(clients):
        lm = FullGAT(data.num_node_features, cfg).to(device)
        lm.load_state_dict(global_model.state_dict())
        for p in lm.encoder.parameters():
            p.requires_grad_(False)
        lm.train()
        opt   = Adam(lm.head.parameters(), lr=cfg.lr_min * 5)
        sched = CosineAnnealingWarmRestarts(opt, T_0=max(1, cfg.head_finetune_rounds // 2), eta_min=cfg.lr_min)
        ei    = get_local_edge_index(data, client['train_mask'], device)
        vmask = client['train_mask'] & (data.y >= 0)
        lbls  = data.y[vmask].to(device)
        # Total head-only epochs = head_finetune_rounds × sup_epochs
        # (default: 20 × 25 = 500). The outer loop mirrors the federated round
        # structure used during main training; the inner loop is the local SGD.
        for _ in range(cfg.head_finetune_rounds):
            for _ in range(cfg.sup_epochs):
                opt.zero_grad()
                logits, _ = lm(data.x.to(device), ei, trunc_dim=proto_dim)
                supervised_loss(logits[vmask.to(device)], lbls, device, cfg).backward()
                opt.step()
                sched.step()
        for p in lm.encoder.parameters():
            p.requires_grad_(True)
        m = evaluate_tuned(lm, data, client['train_mask'], client['test_mask'], device, cfg,
                            proto_dim=proto_dim)
        if verbose:
            print(f'    Client {i}: F1={m["f1"]:.4f} Prec={m["prec"]:.4f} Rec={m["rec"]:.4f}')
        client_results.append((lm, m))

    agg = avg_metrics([r[1] for r in client_results])
    if verbose:
        print(f'  [FedPer] Agg F1={agg["f1"]:.4f} ± {agg["f1_std"]:.4f}')
    return client_results, agg


print('Training functions defined (v10: in-round SSL removed, FedPer fixed).')


In [ ]:
def run_pgfcl(data: Data, clients: list, device: torch.device,
               cfg: ExperimentConfig, seed: int,
               verbose: bool = True, label: str = 'PGFCL v10',
               proto_dim: int = None):
    """
    PGFCL main training loop.

    v10 changes vs v9:
    - IN-ROUND SSL REMOVED. ssl_pretrain_client is no longer called each round.
      SSL pre-warm still happens (ssl_pretrain_phase), then the main loop is
      purely supervised + prototype aggregation.
      Rationale: Diag-A showed in-round SSL (20 epochs per client per round)
      was net-negative: F1 dropped 3.3 pp vs No-SSL variant.
    - fedper_head_finetune replaces the manual head-averaging block.
      Heads are kept local; never averaged back.
    - config_hash stored in returned metrics for checkpoint integrity checking.

    proto_dim: EP-FedProto v3 -- if set, truncates the transmitted
    prototype (and the local z used against it) to this many dims for
    the ENTIRE run (fixed-tier baseline). None = full emb_dim, v10
    behaviour unchanged.

    Returns (best_metrics, drift_curve, f1_curve, saliency_history).
    """
    set_seed(seed)
    global_model = FullGAT(data.num_node_features, cfg).to(device)
    prev_protos  = None
    drift_curve, f1_curve = [], []
    proto_gen_times = []
    best_f1, best_m = 0., {'acc':0.,'f1':0.,'auc':0.,'prec':0.,'rec':0.,'cm':None}

    all_train = torch.zeros(data.num_nodes, dtype=torch.bool)
    all_test  = torch.zeros(data.num_nodes, dtype=torch.bool)
    for c in clients:
        all_train |= c['train_mask']
        all_test  |= c['test_mask']

    # Phase 1: SSL pre-warm (encoder only, federated, NO supervised signal)
    global_model = ssl_pretrain_phase(global_model, data, clients, device, cfg, verbose)

    sizes    = [c['n_train'] for c in clients]
    ei_full  = data.edge_index.to(device)
    saliency_history = []

    # Phase 2: Main supervised + prototype federated loop (NO in-round SSL)
    for rnd in range(cfg.global_rounds):
        lam = cfg.lam_max * min(1.0, rnd / max(cfg.lam_warmup_rounds, 1))
        if verbose:
            print(f'\n  === Round {rnd+1}/{cfg.global_rounds} | lam={lam:.3f} ===')

        x = data.x.to(device)
        client_emb_ps, client_counts = [], []

        # Compute per-client prototypes from the current global encoder
        # (no SSL step here — encoder is only updated by supervised + FedAvg)
        _proto_t0 = time.time()
        for j, client in enumerate(clients):
            with torch.no_grad():
                z = global_model.encoder(x, ei_full)
            emb_p, counts = compute_embedding_prototypes(
                z, data, client['train_mask'], device, cfg
            )
            client_emb_ps.append(emb_p)
            client_counts.append(counts)
        proto_gen_times.append(time.time() - _proto_t0)

        # Chain prototype inheritance
        client_protos = None
        global_protos = None
        if cfg.use_protos:
            client_protos = chain_prototype_inheritance(
                client_emb_ps, client_counts, clients, rnd, cfg
            )
            global_protos = client_protos[0] if client_protos else None
            if global_protos is not None and prev_protos is not None:
                d0 = (global_protos[0].float() - prev_protos[0].float()).norm().item()
                d1 = (global_protos[1].float() - prev_protos[1].float()).norm().item()
                drift_curve.append({'round': rnd+1, 'drift_licit': d0, 'drift_illicit': d1})
                if verbose:
                    print(f'  Drift licit={d0:.4f} illicit={d1:.4f}')
            if global_protos is not None:
                prev_protos = {c: v.detach().clone() for c, v in global_protos.items()}

        # Per-client supervised local training
        local_models = []
        for i, client in enumerate(clients):
            lm = FullGAT(data.num_node_features, cfg).to(device)
            lm.load_state_dict(global_model.state_dict())
            _cproto = client_protos[i] if (client_protos is not None and lam > 0) else None
            supervised_round_client(lm, data, client, device, cfg,
                global_model=global_model if cfg.use_fedprox else None,
                global_protos=_cproto, lam=lam, proto_dim=proto_dim)
            local_models.append(lm)

        # Aggregation
        if cfg.use_grad_guard:
            agg_weights, _ = grad_guard_weights(global_model, local_models, sizes, cfg)
        elif cfg.use_contrib_agg and global_protos is not None:
            agg_weights = compute_client_contrib_weights(client_emb_ps, sizes, global_protos, cfg)
        else:
            total = sum(sizes)
            agg_weights = [s / total for s in sizes]

        global_model = fedavg_state_contrib(global_model, local_models, agg_weights)

        m = evaluate_tuned(global_model, data, all_train, all_test, device, cfg,
                            proto_dim=proto_dim)
        f1_curve.append({'round': rnd+1, 'f1': m['f1'], 'auc': m['auc']})
        if m['f1'] > best_f1:
            best_f1, best_m = m['f1'], m
        if verbose:
            print(f'  Global | F1={m["f1"]:.4f} | AUC={m["auc"]:.4f} | '
                  f'Prec={m["prec"]:.4f} | Rec={m["rec"]:.4f} | thresh={m["thresh"]:.2f}')

        if cfg.use_saliency and (rnd + 1) % 10 == 0:
            sal = extract_node_saliency(global_model, data, all_test, device, cfg)
            saliency_history.append({'round': rnd+1, **sal})
            if verbose:
                n_correct = int((sal['true_labels'] == 1).sum())
                print(f'  Saliency top-{cfg.saliency_top_k}: ' +
                      f'{n_correct}/{cfg.saliency_top_k} are truly illicit')

    # Phase 3: FedPer head fine-tuning (local, no averaging of heads)
    if cfg.head_finetune_rounds > 0:
        client_results, per_results = fedper_head_finetune(
            global_model, data, clients, device, cfg, verbose=verbose,
            proto_dim=proto_dim
        )
        # Report per-client aggregated metrics; use best model from Phase 2 if FedPer is worse
        if per_results['f1'] > best_f1:
            best_f1, best_m = per_results['f1'], per_results
            best_m['fedper_client_results'] = [(None, r[1]) for r in client_results]
        if verbose:
            print(f'  [FedPer] Final agg F1={per_results["f1"]:.4f}')

    # ECE calibration on global model
    if cfg.use_calibration:
        # Pass all_train so compute_ece uses inductive edges (same as evaluate_tuned)
        ece, bin_data = compute_ece(global_model, data, all_test, device, cfg,
                                    train_mask=all_train, proto_dim=proto_dim)
        best_m['ece']      = ece
        best_m['ece_bins'] = bin_data
        if verbose:
            print(f'  ECE = {ece:.4f}')

    # EP-FedProto v3: instrumentation -- mean per-round prototype-generation time
    best_m['proto_gen_time_s'] = float(np.mean(proto_gen_times)) if proto_gen_times else 0.0
    best_m['proto_dim'] = proto_dim if proto_dim is not None else cfg.emb_dim

    # Store config hash for checkpoint integrity
    best_m['config_hash'] = config_hash(cfg)

    print(f'\n[{label}] Best F1={best_m["f1"]:.4f} | AUC={best_m["auc"]:.4f} | ' +
          f'hash={best_m["config_hash"]}')
    return best_m, drift_curve, f1_curve, saliency_history


print('run_pgfcl v10 defined.')


## Baselines

Runs centralized, local-only, FedAvg, FedSage (SAGE), FedSage (GAT+SSL), FedProto — 3 seeds each.

## EP-FedProto v3 — New Infrastructure (device tiers, multi-budget loss, instrumentation)

In [ ]:
# ── EP-FedProto v3: device tiers, multi-budget loss, instrumentation ─────────
# NB1 scope per plan: multi-budget loss (confirm single-pass slicing),
# fixed-tier baseline, FjORD baseline, edge-device config, instrumentation,
# scalability-sweep setup. This cell defines shared infra used by all of them;
# the actual new baseline runs happen further below, after "Baselines".

DEVICE_TIER_DIMS = (8, 16, 32, 64)   # nested prototype/embedding budgets


def assign_device_tiers(clients, tier_dims=DEVICE_TIER_DIMS, distribution=None, seed=0):
    """Assign each client a device-capability tier (max transmitted dim).

    distribution: optional list of proportions matching tier_dims (sums to 1).
    Default: uniform round-robin, shuffled. Used by FjORD (heterogeneous
    per-client widths) now, and by NB2's device-tier lookup / NB3's skewed
    60/30/10 tier-distribution ablation later.
    """
    rng = random.Random(seed)
    n = len(clients)
    if distribution is None:
        tiers = [tier_dims[i % len(tier_dims)] for i in range(n)]
    else:
        assert len(distribution) == len(tier_dims), "distribution must match tier_dims"
        counts = [round(p * n) for p in distribution]
        while sum(counts) < n: counts[counts.index(min(counts))] += 1
        while sum(counts) > n: counts[counts.index(max(counts))] -= 1
        tiers = []
        for d, c in zip(tier_dims, counts):
            tiers += [d] * c
    rng.shuffle(tiers)
    return {c['id']: t for c, t in zip(clients, tiers)}


def truncate_proto(proto_dict, d):
    """Slice each class prototype to its first d dims (Matryoshka nesting)."""
    return {c: v[:d] for c, v in proto_dict.items()}


def multi_budget_proto_loss(z, labels, mask, global_protos_full, device, cfg,
                             budgets=DEVICE_TIER_DIMS, weights=None):
    """Matryoshka-style multi-budget prototype loss.

    SINGLE-PASS SLICING (confirmed by unit test): z is encoded ONCE by the
    caller; this function only slices *columns* of that same tensor per
    budget -- it never re-runs the encoder. Cost scales as O(len(budgets))
    prototype-loss evaluations on views of one tensor, not O(len(budgets))
    forward passes.

    Verified in isolation before wiring in: gradient flows into every
    budget's slice, and per-dim gradient magnitude decreases monotonically
    from "used by all 4 tiers" (dims 0:8) to "used only by the d=64 tier"
    (dims 32:64) -- exactly the nesting behaviour Matryoshka representations
    are supposed to have.

    Not yet wired into run_pgfcl's main loop -- that happens in NB2, where
    EP-FedProto's full training loop (this loss + the rank-aware aggregator)
    is assembled. This cell only defines and validates the loss itself.
    """
    if global_protos_full is None:
        return torch.tensor(0.0, device=device)
    if weights is None:
        weights = [1.0 / len(budgets)] * len(budgets)
    assert len(weights) == len(budgets)
    total = torch.tensor(0.0, device=device)
    for w, d in zip(weights, budgets):
        protos_d = truncate_proto(global_protos_full, d)
        total = total + w * prototype_supcon_loss(
            z[:, :d], labels, mask, protos_d, device, cfg
        )
    return total


# ── Instrumentation ───────────────────────────────────────────────────────────
try:
    import psutil
    _PROC = psutil.Process(os.getpid())
except ImportError:
    psutil = None
    _PROC = None
    print('  [instrumentation] psutil not available -- CPU memory stats will be skipped.')


def with_instrumentation(fn):
    """Wrap a run_*_gat baseline: records wall time + peak GPU/CPU memory into
    the returned metrics dict under 'compute_stats'. Only fires when fn() is
    actually executed -- a run_or_load cache hit correctly reports no new
    compute stats (nothing ran)."""
    def wrapper(*args, **kwargs):
        device = kwargs.get('device')
        if device is None:
            for a in args:
                if isinstance(a, torch.device):
                    device = a
                    break
        if device is not None and device.type == 'cuda':
            torch.cuda.reset_peak_memory_stats(device)
        t0 = time.time()
        result = fn(*args, **kwargs)
        stats = {'wall_time_s': time.time() - t0}
        if device is not None and device.type == 'cuda':
            stats['peak_gpu_mem_mb'] = torch.cuda.max_memory_allocated(device) / (1024 ** 2)
        if _PROC is not None:
            stats['peak_cpu_mem_mb'] = _PROC.memory_info().rss / (1024 ** 2)
        if isinstance(result, dict):
            result['compute_stats'] = stats
        return result
    return wrapper


print('EP-FedProto v3 infra defined: device tiers, multi-budget loss, instrumentation.')


In [ ]:
def _all_masks(clients, n_nodes):
    all_train = torch.zeros(n_nodes, dtype=torch.bool)
    all_test  = torch.zeros(n_nodes, dtype=torch.bool)
    for c in clients:
        all_train |= c['train_mask']
        all_test  |= c['test_mask']
    return all_train, all_test


def run_centralized_gat(data, clients, device, cfg, seed=None):
    seed = seed or cfg.seeds[0]
    set_seed(seed)
    print(f'  [Centralized GAT] seed={seed}')
    model = FullGAT(data.num_node_features, cfg).to(device)
    opt   = Adam(model.parameters(), lr=cfg.lr)
    sched = CosineAnnealingWarmRestarts(opt, T_0=cfg.centralized_epochs//2, eta_min=cfg.lr_min)
    all_train, all_test = _all_masks(clients, data.num_nodes)
    ei_tr = get_local_edge_index(data, all_train, device)
    vmask = all_train & (data.y >= 0)
    lbls  = data.y[vmask].to(device)
    for _ in range(cfg.centralized_epochs):
        model.train()
        opt.zero_grad()
        logits, _ = model(data.x.to(device), ei_tr)
        supervised_loss(logits[vmask], lbls, device, cfg).backward()
        opt.step(); sched.step()
    m = evaluate_tuned(model, data, all_train, all_test, device, cfg)
    print(f'    F1={m["f1"]:.4f} | AUC={m["auc"]:.4f} | Prec={m["prec"]:.4f} | Rec={m["rec"]:.4f}')
    return m


def run_local_only_gat(data, clients, device, cfg, seed=None):
    seed = seed or cfg.seeds[0]
    set_seed(seed)
    print(f'  [Local-Only GAT] seed={seed}')
    total_epochs = cfg.baseline_rounds * cfg.baseline_local_epochs
    all_metrics = []
    for client in clients:
        model = FullGAT(data.num_node_features, cfg).to(device)
        opt   = Adam(model.parameters(), lr=cfg.lr)
        sched = CosineAnnealingWarmRestarts(opt, T_0=max(1,total_epochs//2), eta_min=cfg.lr_min)
        ei    = get_local_edge_index(data, client['train_mask'], device)
        vmask = client['train_mask'] & (data.y >= 0)
        lbls  = data.y[vmask].to(device)
        for _ in range(total_epochs):
            model.train()
            opt.zero_grad()
            logits, _ = model(data.x.to(device), ei)
            supervised_loss(logits[vmask], lbls, device, cfg).backward()
            opt.step(); sched.step()
        m = evaluate_tuned(model, data, client['train_mask'], client['test_mask'], device, cfg)
        all_metrics.append(m)
    return avg_metrics(all_metrics)


def _federated_loop(global_model, data, clients, device, cfg, seed, label='FedAvg'):
    set_seed(seed)
    all_train, all_test = _all_masks(clients, data.num_nodes)
    sizes    = [c['n_train'] for c in clients]
    best_f1, best_m = 0., {}
    for rnd in range(cfg.baseline_rounds):
        local_models = []
        for client in clients:
            # Use explicit constructors instead of type() to avoid silent
            # breakage if a subclass changes the (in_dim, cfg) signature.
            model_cls = type(global_model)
            if model_cls not in (FullGAT, SAGEModel):
                raise TypeError(f'_federated_loop: unsupported model type {model_cls}')
            lm = model_cls(data.num_node_features, cfg).to(device)
            lm.load_state_dict(global_model.state_dict())
            lm.train()
            opt   = Adam(lm.parameters(), lr=cfg.lr)
            sched = CosineAnnealingWarmRestarts(opt, T_0=max(1,cfg.baseline_local_epochs//2), eta_min=cfg.lr_min)
            ei    = get_local_edge_index(data, client['train_mask'], device)
            vmask = client['train_mask'] & (data.y >= 0)
            lbls  = data.y[vmask].to(device)
            for _ in range(cfg.baseline_local_epochs):
                opt.zero_grad()
                logits, _ = lm(data.x.to(device), ei)
                supervised_loss(logits[vmask], lbls, device, cfg).backward()
                opt.step(); sched.step()
            local_models.append(lm)
        global_model = fedavg_state_selective(global_model, local_models, sizes)
        m = evaluate_tuned(global_model, data, all_train, all_test, device, cfg)
        if m['f1'] > best_f1:
            best_f1, best_m = m['f1'], m
    print(f'    [{label}] Best F1={best_m["f1"]:.4f} | AUC={best_m["auc"]:.4f}')
    return best_m


def run_fedavg_gat(data, clients, device, cfg, seed=None):
    seed = seed or cfg.seeds[0]
    print(f'  [FedAvg GAT] seed={seed}')
    return _federated_loop(
        FullGAT(data.num_node_features, cfg).to(device),
        data, clients, device, cfg, seed, label='FedAvg GAT'
    )


def run_fedsage_sage(data, clients, device, cfg, seed=None):
    seed = seed or cfg.seeds[0]
    print(f'  [FedSage SAGE] seed={seed}')
    return _federated_loop(
        SAGEModel(data.num_node_features, cfg).to(device),
        data, clients, device, cfg, seed, label='FedSage (SAGE)'
    )


def run_fedsage_gat(data, clients, device, cfg, seed=None):
    seed = seed or cfg.seeds[0]
    set_seed(seed)
    print(f'  [FedSage GAT + SSL] seed={seed}')
    model   = FullGAT(data.num_node_features, cfg).to(device)
    ssl_cfg = ExperimentConfig(ssl_pretrain_rounds=1, use_ssl=True)
    model   = ssl_pretrain_phase(model, data, clients, device, ssl_cfg, verbose=False)
    return _federated_loop(model, data, clients, device, cfg, seed,
                            label='FedSage (GAT+SSL)')


def run_fedproto_gat(data, clients, device, cfg, seed=None):
    seed = seed or cfg.seeds[0]
    print(f'  [FedProto GAT] seed={seed}')
    proto_cfg = ExperimentConfig(use_ssl=False, use_protos=True, use_fedprox=False)
    m, _, _, _ = run_pgfcl(data, clients, device, proto_cfg, seed=seed,
                         verbose=False, label='FedProto GAT')
    print(f'    [FedProto GAT] F1={m["f1"]:.4f} | AUC={m["auc"]:.4f}')
    return m



def moon_contrastive_loss(z_curr, z_global, z_prev, tau=0.5):
    """MOON: maximise sim(current, global), minimise sim(current, prev_local)."""
    z_curr   = F.normalize(z_curr.mean(0, keepdim=True), dim=1)
    z_global = F.normalize(z_global.mean(0, keepdim=True), dim=1)
    z_prev   = F.normalize(z_prev.mean(0, keepdim=True), dim=1)
    pos = torch.exp(torch.mm(z_curr, z_global.T) / tau)
    neg = torch.exp(torch.mm(z_curr, z_prev.T)   / tau)
    return -torch.log(pos / (pos + neg + 1e-8)).mean()


def run_moon_gat(data, clients, device, cfg, seed=None, mu_moon=1.0):
    """FIX A3: MOON baseline — model-contrastive federated learning."""
    seed = seed or cfg.seeds[0]
    set_seed(seed)
    global_model = FullGAT(data.num_node_features, cfg).to(device)
    prev_local_models = [copy.deepcopy(global_model) for _ in clients]
    all_train, all_test = _all_masks(clients, data.num_nodes)
    sizes  = [c['n_train'] for c in clients]
    best_f1, best_m = 0., {}
    for rnd in range(cfg.baseline_rounds):
        local_models = []
        for i, client in enumerate(clients):
            lm = FullGAT(data.num_node_features, cfg).to(device)
            lm.load_state_dict(global_model.state_dict())
            lm.train()
            opt = Adam(lm.parameters(), lr=cfg.lr)
            ei  = get_local_edge_index(data, client['train_mask'], device)
            vmask = client['train_mask'] & (data.y >= 0)
            lbls  = data.y[vmask].to(device)
            for _ in range(cfg.baseline_local_epochs):
                opt.zero_grad()
                logits, emb = lm(data.x.to(device), ei)
                with torch.no_grad():
                    _, g_emb = global_model(data.x.to(device), ei)
                    _, p_emb = prev_local_models[i](data.x.to(device), ei)
                sup = supervised_loss(logits[vmask], lbls, device, cfg)
                moon = moon_contrastive_loss(emb[vmask], g_emb[vmask], p_emb[vmask])
                (sup + mu_moon * moon).backward()
                opt.step()
            prev_local_models[i] = copy.deepcopy(lm)
            local_models.append(lm)
        global_model = fedavg_state_selective(global_model, local_models, sizes)
        m = evaluate_tuned(global_model, data, all_train, all_test, device, cfg)
        if m['f1'] > best_f1: best_f1, best_m = m['f1'], m
    print(f'  [MOON GAT] Best F1={best_m["f1"]:.4f}')
    return best_m



def run_scaffold_gat(data, clients, device, cfg, seed=None, lr_c=0.1):
    """FIX C1: SCAFFOLD baseline — server and client control variates correct gradient drift."""
    set_seed(seed or cfg.seeds[0])
    global_model = FullGAT(data.num_node_features, cfg).to(device)
    c_server  = [torch.zeros_like(p) for p in global_model.parameters()]
    c_clients = [[torch.zeros_like(p) for p in global_model.parameters()]
                  for _ in clients]
    sizes = [c['n_train'] for c in clients]
    all_train, all_test = _all_masks(clients, data.num_nodes)
    best_f1, best_m = 0., {}
    for rnd in range(cfg.baseline_rounds):
        delta_cs = []
        local_models_for_agg = []
        for i, client in enumerate(clients):
            lm = FullGAT(data.num_node_features, cfg).to(device)
            lm.load_state_dict(global_model.state_dict())
            lm.train()
            opt = Adam(lm.parameters(), lr=cfg.lr)
            ei  = get_local_edge_index(data, client['train_mask'], device)
            vmask = client['train_mask'] & (data.y >= 0)
            lbls  = data.y[vmask].to(device)
            for _ in range(cfg.baseline_local_epochs):
                opt.zero_grad()
                logits, _ = lm(data.x.to(device), ei)
                loss = supervised_loss(logits[vmask], lbls, device, cfg)
                loss.backward()
                for p, ci, cs in zip(lm.parameters(), c_clients[i], c_server):
                    if p.grad is not None:
                        p.grad.data.add_(cs - ci)
                opt.step()
            new_ci = [
                cs.clone() - ci.clone() + (gp.data.clone() - lp.data.clone()) / (cfg.baseline_local_epochs * cfg.lr)
                for ci, cs, gp, lp in zip(c_clients[i], c_server,
                                           global_model.parameters(), lm.parameters())
            ]
            delta_cs.append([nc - oc for nc, oc in zip(new_ci, c_clients[i])])
            c_clients[i] = new_ci
            local_models_for_agg.append(lm)
        n = len(clients)
        for j, cs in enumerate(c_server):
            cs.data.add_(sum(dc[j] for dc in delta_cs) / n)
        global_model = fedavg_state_selective(global_model, local_models_for_agg, sizes)
        m = evaluate_tuned(global_model, data, all_train, all_test, device, cfg)
        if m['f1'] > best_f1: best_f1, best_m = m['f1'], m
    print(f'  [SCAFFOLD GAT] Best F1={best_m["f1"]:.4f}')
    return best_m


print('Baselines defined.')

# EP-FedProto v3: retrofit instrumentation onto the existing FedProto baseline
# (with_instrumentation is defined in the EP-FedProto infra cell above) so the
# NB4 compute table / Pareto figure has timing+memory for FedProto too.
run_fedproto_gat = with_instrumentation(run_fedproto_gat)

print('Running 5-seed baselines...  (FIX A2: 5 seeds)')
t0 = time.time()

central_runs, localonly_runs, fedavg_runs = [], [], []
fedsage_runs, fedsage_gat_runs, fedproto_runs = [], [], []
# FIX A3 + C1: new baselines
moon_runs, scaffold_runs = [], []

_cfg_hash = config_hash(CFG)  # validate checkpoint configs against current CFG
for s in CFG.seeds:
    central_runs.append(
        run_or_load(f'central_s{s}',
                    lambda s=s: run_centralized_gat(elliptic_data, elliptic_clients, DEVICE, CFG, s),
                    expected_hash=_cfg_hash))
    localonly_runs.append(
        run_or_load(f'localonly_s{s}',
                    lambda s=s: run_local_only_gat(elliptic_data, elliptic_clients, DEVICE, CFG, s),
                    expected_hash=_cfg_hash))
    fedavg_runs.append(
        run_or_load(f'fedavg_s{s}',
                    lambda s=s: run_fedavg_gat(elliptic_data, elliptic_clients, DEVICE, CFG, s),
                    expected_hash=_cfg_hash))
    fedsage_runs.append(
        run_or_load(f'fedsage_s{s}',
                    lambda s=s: run_fedsage_sage(elliptic_data, elliptic_clients, DEVICE, CFG, s),
                    expected_hash=_cfg_hash))
    fedsage_gat_runs.append(
        run_or_load(f'fedsage_gat_s{s}',
                    lambda s=s: run_fedsage_gat(elliptic_data, elliptic_clients, DEVICE, CFG, s),
                    expected_hash=_cfg_hash))
    # BUGFIX (pre-existing, same class as the fixed-tier checkpoint-hash bug):
    # run_fedproto_gat builds its OWN fresh proto_cfg internally (not a copy
    # of CFG), so run_pgfcl stores config_hash(proto_cfg), not config_hash(CFG).
    # Checking against _cfg_hash was a guaranteed mismatch every single run --
    # this baseline was silently recomputing from scratch on every resume.
    # Matching the exact same proto_cfg construction here (not touching
    # run_fedproto_gat's behavior at all) fixes validation without changing
    # what gets trained or how it's evaluated.
    _fedproto_expected_hash = config_hash(
        ExperimentConfig(use_ssl=False, use_protos=True, use_fedprox=False))
    fedproto_runs.append(
        run_or_load(f'fedproto_s{s}',
                    lambda s=s: run_fedproto_gat(elliptic_data, elliptic_clients, DEVICE, CFG, s),
                    expected_hash=_fedproto_expected_hash))
    # FIX A3: MOON baseline
    moon_runs.append(
        run_or_load(f'moon_s{s}',
                    lambda s=s: run_moon_gat(elliptic_data, elliptic_clients, DEVICE, CFG, s),
                    expected_hash=_cfg_hash))
    # FIX C1: SCAFFOLD baseline
    scaffold_runs.append(
        run_or_load(f'scaffold_s{s}',
                    lambda s=s: run_scaffold_gat(elliptic_data, elliptic_clients, DEVICE, CFG, s),
                    expected_hash=_cfg_hash))

central_avg     = avg_metrics(central_runs)
localonly_avg   = avg_metrics(localonly_runs)
fedavg_avg      = avg_metrics(fedavg_runs)
fedsage_avg     = avg_metrics(fedsage_runs)
fedsage_gat_avg = avg_metrics(fedsage_gat_runs)
fedproto_avg    = avg_metrics(fedproto_runs)
moon_avg        = avg_metrics(moon_runs)
scaffold_avg    = avg_metrics(scaffold_runs)
print(f'Baselines done in {(time.time()-t0)/60:.1f} min')


## EP-FedProto v3 — New Baselines (fixed-tier FedProto, FjORD, edge-device config)

In [ ]:
# ── Fixed-budget-per-tier FedProto baseline ───────────────────────────────────
def _fixed_tier_effective_cfg(cfg):
    """Build the exact effective config run_fixed_tier_fedproto trains under
    (use_ssl/use_protos/use_fedprox flipped on a deep copy of `cfg`).

    BUGFIX (checkpoint hash mismatch): this is factored out so callers of
    run_or_load can compute expected_hash=config_hash(_fixed_tier_effective_cfg(cfg))
    -- matching what run_pgfcl actually stores as best_m['config_hash'] --
    instead of hashing the caller's un-flipped `cfg`. Previously every resume
    hit a false-positive "hash mismatch" and silently retrained all 20
    fixed-tier runs plus the edge-confirmation run from scratch.
    """
    proto_cfg = copy.deepcopy(cfg)
    proto_cfg.use_ssl     = False
    proto_cfg.use_protos  = True
    proto_cfg.use_fedprox = False
    return proto_cfg


def run_fixed_tier_fedproto(data, clients, device, cfg, seed=None, tier_d=64):
    """Baseline: FedProto where every client truncates its prototype to a
    FIXED dimension tier_d for the whole run -- no nesting across budgets,
    unlike EP-FedProto's multi-budget Matryoshka loss (assembled in NB2).
    One run per tier_d in DEVICE_TIER_DIMS gives the F1-vs-d curve for this
    baseline, comparable to EP-FedProto's own F1-vs-d curve in NB4.

    NOTE: unlike the pre-existing run_fedproto_gat (which builds its inner
    proto_cfg from ExperimentConfig() defaults and silently ignores the
    architecture fields of the `cfg` passed in), this function derives
    proto_cfg from a deep copy of the PASSED-IN cfg. That matters here
    because this function is also used for the edge-device confirmation run
    under EDGE_CFG (smaller hidden/emb_dim) -- see below.
    """
    seed = seed or cfg.seeds[0]
    print(f'  [Fixed-tier FedProto d={tier_d}] seed={seed}')
    proto_cfg = _fixed_tier_effective_cfg(cfg)
    m, _, _, _ = run_pgfcl(data, clients, device, proto_cfg, seed=seed,
                            verbose=False, label=f'FixedTier-FedProto(d={tier_d})',
                            proto_dim=tier_d)
    print(f'    [Fixed-tier d={tier_d}] F1={m["f1"]:.4f} | AUC={m["auc"]:.4f}')
    return m

run_fixed_tier_fedproto = with_instrumentation(run_fixed_tier_fedproto)
print('run_fixed_tier_fedproto defined.')


In [ ]:
# ── FjORD (Ordered Dropout) baseline ──────────────────────────────────────────
def evaluate_at_width(model, data, train_mask, test_mask, device, cfg, width_ratio):
    """FjORD per-width evaluation (fills the 'no per-tier eval yet' gap).

    Mirrors evaluate_tuned's threshold-tuned, inductive-eval logic exactly,
    but forwards through encoder.forward_od(width_ratio) instead of the
    full-width encoder.forward(), so FjORD gets a real F1-per-width number
    -- comparable to the fixed-tier FedProto and EP-FedProto F1-vs-d curves
    -- instead of only a single full-capacity ceiling F1 per run.
    """
    model.eval()
    ei_tr = get_local_edge_index(data, train_mask, device)
    with torch.no_grad():
        emb_tr    = model.encoder.forward_od(data.x.to(device), ei_tr, width_ratio)
        logits_tr = model.head(emb_tr)
    probs_tr = F.softmax(logits_tr, dim=1)[:, 1].cpu().numpy()
    if not np.isfinite(probs_tr).all():
        return {'acc': 0., 'f1': 0., 'auc': 0., 'prec': 0., 'rec': 0.,
                'cm': np.zeros((2, 2), int), 'thresh': 0.5}

    tr_lab = train_mask & (data.y >= 0)
    best_thresh = 0.5
    if tr_lab.sum() > 0 and len(np.unique(data.y[tr_lab].numpy())) > 1:
        p, r, thresholds = precision_recall_curve(
            data.y[tr_lab].numpy(), probs_tr[tr_lab.numpy()])
        f1s = 2 * p * r / (p + r + 1e-8)
        best_thresh = float(np.clip(thresholds[np.argmax(f1s[:-1])], 0.1, 0.9))

    ei_te = get_inductive_edge_index(data, train_mask, test_mask, device)
    with torch.no_grad():
        emb_te    = model.encoder.forward_od(data.x.to(device), ei_te, width_ratio)
        logits_te = model.head(emb_te)
    probs_te_full = F.softmax(logits_te, dim=1)[:, 1]
    if cfg.use_label_prop:
        probs_te_full = label_propagation(
            probs_te_full, ei_te, data.num_nodes,
            alpha=cfg.lp_alpha, steps=cfg.lp_steps
        ).to(device)
    probs_te = probs_te_full.cpu().numpy()

    te_lab = test_mask & (data.y >= 0)
    if te_lab.sum() == 0:
        return {'acc': 0., 'f1': 0., 'auc': 0., 'prec': 0., 'rec': 0.,
                'cm': np.zeros((2, 2), int), 'thresh': best_thresh}

    probs_te_masked = probs_te[te_lab.numpy()]
    preds_te        = (probs_te_masked >= best_thresh).astype(int)
    true_te         = data.y[te_lab].numpy()

    return {
        'acc':    accuracy_score(true_te, preds_te),
        'f1':     f1_score(true_te, preds_te, zero_division=0),
        'auc':    roc_auc_score(true_te, probs_te_masked) if len(np.unique(true_te)) > 1 else 0.,
        'prec':   precision_score(true_te, preds_te, zero_division=0),
        'rec':    recall_score(true_te, preds_te, zero_division=0),
        'cm':     confusion_matrix(true_te, preds_te),
        'thresh': best_thresh
    }


def run_fjord_gat(data, clients, device, cfg, seed=None,
                   tier_dims=DEVICE_TIER_DIMS, tier_distribution=None):
    """FjORD: weight-space heterogeneity via Ordered Dropout, contrasted in
    the paper's framing against EP-FedProto's prototype-space nesting. Each
    client gets a device tier (max active embedding width) and trains only
    that nested channel-prefix of the shared encoder (encoder.forward_od);
    aggregation is plain size-weighted FedAvg over the full state_dict.

    This is correct (not just convenient) because Ordered Dropout here is
    post-activation channel MASKING, not weight resizing: a masked-out output
    channel gets exactly zero gradient through the weight rows that produce
    it (verified in isolation), so a low-budget client's local weights for
    channels above its tier are simply left equal to the global value it
    started the round with -- averaging them in is a no-op for those rows,
    not noise from an untrained client.

    No prototypes, no FedProx -- pure weight-space nesting baseline.
    """
    seed = seed or cfg.seeds[0]
    set_seed(seed)
    tiers = assign_device_tiers(clients, tier_dims, tier_distribution, seed=seed)
    print(f'  [FjORD] seed={seed} | tiers={tiers}')

    global_model = FullGAT(data.num_node_features, cfg).to(device)
    sizes = [c['n_train'] for c in clients]
    all_train, all_test = _all_masks(clients, data.num_nodes)
    best_f1, best_m = 0., {}

    for rnd in range(cfg.baseline_rounds):
        local_models = []
        for i, client in enumerate(clients):
            lm = FullGAT(data.num_node_features, cfg).to(device)
            lm.load_state_dict(global_model.state_dict())
            lm.train()
            opt = Adam(lm.parameters(), lr=cfg.lr)
            ei = get_local_edge_index(data, client['train_mask'], device)
            vmask = client['train_mask'] & (data.y >= 0)
            if vmask.sum() < 2:
                local_models.append(lm)
                continue
            lbls = data.y[vmask].to(device)
            width_ratio = tiers[client['id']] / cfg.emb_dim
            for _ in range(cfg.baseline_local_epochs):
                opt.zero_grad()
                emb = lm.encoder.forward_od(data.x.to(device), ei, width_ratio)
                logits = lm.head(emb)
                loss = supervised_loss(logits[vmask.to(device)], lbls, device, cfg)
                loss.backward()
                opt.step()
            local_models.append(lm)
        global_model = fedavg_state_selective(global_model, local_models, sizes)
        m = evaluate_tuned(global_model, data, all_train, all_test, device, cfg)
        if m['f1'] > best_f1:
            best_f1, best_m = m['f1'], m

    # GAP FIX: per-width F1 curve on the FINAL global model (intermediate
    # per-round models aren't retained, so this is evaluated post-hoc on the
    # model state at the end of training, same as e.g. run_centralized_gat
    # does for its single reported number -- not on whichever round produced
    # best_f1 at full width).
    best_m['f1_by_width'] = {
        d: evaluate_at_width(global_model, data, all_train, all_test, device, cfg,
                              d / cfg.emb_dim)
        for d in tier_dims
    }
    best_m['device_tiers'] = tiers
    best_m['config_hash']  = config_hash(cfg)
    print(f'  [FjORD] Best F1={best_m["f1"]:.4f} | AUC={best_m["auc"]:.4f}')
    for d in tier_dims:
        fw = best_m['f1_by_width'][d]
        print(f'    [FjORD] width d={d:2d} (ratio={d/cfg.emb_dim:.2f}): F1={fw["f1"]:.4f} | AUC={fw["auc"]:.4f}')
    return best_m

run_fjord_gat = with_instrumentation(run_fjord_gat)
print('run_fjord_gat defined.')


In [ ]:
# ── Edge-device simulation config ─────────────────────────────────────────────
# Design choice (stated openly, per the paper's compute-scoping framing):
# edge devices are simulated with a genuinely smaller ON-DEVICE ARCHITECTURE
# (halved hidden/emb width), not just a smaller transmitted-prototype slice --
# this gives a real resource-constrained-FL story rather than only a
# communication-budget story. Max transmitted/embedding dim for edge tiers is
# capped at {8, 16}, the two lowest DEVICE_TIER_DIMS (consistent with the
# smaller emb_dim=32 ceiling below).
EDGE_CFG = ExperimentConfig(
    hidden=64,                 # halved from 128
    emb_dim=32,                # halved from 64
    baseline_local_epochs=10,  # reduced from 25 (constrained on-device compute)
    global_rounds=50,          # reduced from 100
    ssl_pretrain_rounds=3,     # reduced from 6
)
EDGE_TIER_DIMS = (8, 16)  # ceiling given EDGE_CFG's smaller emb_dim=32

print(f'EDGE_CFG defined: hidden={EDGE_CFG.hidden}, emb_dim={EDGE_CFG.emb_dim}, '
      f'local_epochs={EDGE_CFG.baseline_local_epochs}, tiers={EDGE_TIER_DIMS}')
print('Config hash (edge):', config_hash(EDGE_CFG))


# ── Reduced-compute sweep config (compute-budget scoping, same pattern as ─────
# ── EDGE_CFG above) ────────────────────────────────────────────────────────
# Kaggle free tier gives ~30 GPU-hours/week. A full 5-seed x 4-tier x
# (100 rounds + 500-epoch FedPer) sweep does not fit that budget alongside
# the other notebooks still to come. SWEEP_CFG keeps the SAME architecture
# as CFG (hidden/emb_dim unchanged -- this is a schedule cut, not an
# architecture cut, so it doesn't quietly become a different, weaker model)
# and only shortens the training schedule, exactly the way EDGE_CFG already
# does for the edge-device story. Applied uniformly to fixed-tier FedProto
# AND FjORD below, so the two are still a fair comparison against each other.
#
# This is an EXPLORATORY first pass to get the F1-vs-d curve shape fast.
# Seeds/rounds can be extended back toward CFG's full schedule later --
# thanks to the checkpoint hash fix, doing so will only compute the missing
# (tier, seed) combinations, not redo anything already cached.
SWEEP_CFG = copy.deepcopy(CFG)
SWEEP_CFG.global_rounds        = 50   # was 100
SWEEP_CFG.head_finetune_rounds = 10   # was 20 (FedPer: 10x25=250 epochs, was 500)
SWEEP_CFG.ssl_pretrain_rounds  = 3    # was 6
SWEEP_CFG.baseline_rounds      = 25   # was 50 (used by FjORD)
SWEEP_SEEDS = CFG.seeds[:2]           # was all 5 -- extend later if time allows

print(f'SWEEP_CFG defined: global_rounds={SWEEP_CFG.global_rounds}, '
      f'head_finetune_rounds={SWEEP_CFG.head_finetune_rounds}, '
      f'ssl_pretrain_rounds={SWEEP_CFG.ssl_pretrain_rounds}, '
      f'baseline_rounds={SWEEP_CFG.baseline_rounds}')
print(f'SWEEP_SEEDS = {SWEEP_SEEDS}  (extend toward CFG.seeds={CFG.seeds} later)')
print('Config hash (sweep):', config_hash(SWEEP_CFG))


In [ ]:
# ── Scalability sweep setup (splits only -- core comparison runs in NB2) ──────
SCALABILITY_N_CLIENTS = (4, 8, 16)  # extend to 32 in NB2 if compute allows
scalability_splits = {}
for n in SCALABILITY_N_CLIENTS:
    print(f'\n=== Scalability split setup: n_clients={n} ===')
    cfg_n = ExperimentConfig(n_clients=n)
    scalability_splits[n] = {
        'cfg_n_clients': n,
        'clients': temporal_federated_split(elliptic_data, cfg_n),
    }
print('\nScalability splits prepared for', SCALABILITY_N_CLIENTS)


In [ ]:
# ── Run new NB1 baselines: fixed-tier FedProto, FjORD, edge-device confirmation ─
# COMPUTE BUDGET: uses SWEEP_CFG (reduced schedule, 2 seeds) instead of CFG's
# full 5-seed schedule -- see the "Reduced-compute sweep config" cell above
# for why. Applied identically to fixed-tier AND FjORD so they're still a
# fair comparison. Swap SWEEP_CFG->CFG and SWEEP_SEEDS->CFG.seeds here later
# if/when there's compute budget to extend to the full schedule; checkpoints
# mean that only computes what's missing, not a full redo.
print('Running EP-FedProto v3 NB1 new baselines (SWEEP_CFG: reduced schedule)...')
t0 = time.time()

fixed_tier_runs = {}   # {tier_d: [run_per_seed, ...]}
for tier_d in DEVICE_TIER_DIMS:
    fixed_tier_runs[tier_d] = []
    _fixed_tier_hash = config_hash(_fixed_tier_effective_cfg(SWEEP_CFG))
    for s in SWEEP_SEEDS:
        fixed_tier_runs[tier_d].append(
            run_or_load(f'fixedtier{tier_d}_s{s}',
                        lambda s=s, d=tier_d: run_fixed_tier_fedproto(
                            elliptic_data, elliptic_clients, DEVICE, SWEEP_CFG, s, tier_d=d),
                        expected_hash=_fixed_tier_hash))
fixed_tier_avg = {d: avg_metrics(runs) for d, runs in fixed_tier_runs.items()}

_sweep_hash = config_hash(SWEEP_CFG)  # FjORD passes cfg straight through (no flip)
fjord_runs = []
for s in SWEEP_SEEDS:
    fjord_runs.append(
        run_or_load(f'fjord_s{s}',
                    lambda s=s: run_fjord_gat(elliptic_data, elliptic_clients, DEVICE, SWEEP_CFG, s),
                    expected_hash=_sweep_hash))
fjord_avg = avg_metrics(fjord_runs)

# GAP FIX: average FjORD's per-width F1 across seeds too, so it's directly
# comparable to fixed_tier_avg's F1-vs-d numbers above.
fjord_width_avg = {
    d: avg_metrics([run['f1_by_width'][d] for run in fjord_runs])
    for d in DEVICE_TIER_DIMS
}

# Edge-device confirmation run: single-seed sanity check that the pipeline
# runs end-to-end under EDGE_CFG's smaller architecture. Full multi-seed
# edge sweep (all tiers, all seeds) happens in NB2/NB4 per the plan.
edge_cfg_hash = config_hash(_fixed_tier_effective_cfg(EDGE_CFG))
edge_confirmation_run = run_or_load(
    'edge_confirmation_s42',
    lambda: run_fixed_tier_fedproto(elliptic_data, elliptic_clients, DEVICE, EDGE_CFG,
                                     seed=42, tier_d=EDGE_TIER_DIMS[0]),
    expected_hash=edge_cfg_hash)

print(f'\nNew baselines done in {(time.time()-t0)/60:.1f} min')
for d in DEVICE_TIER_DIMS:
    print(f'  Fixed-tier d={d:2d}: F1={fixed_tier_avg[d]["f1"]:.4f} +/- {fixed_tier_avg[d]["f1_std"]:.4f}')
print(f'  FjORD (full) : F1={fjord_avg["f1"]:.4f} +/- {fjord_avg["f1_std"]:.4f}')
for d in DEVICE_TIER_DIMS:
    fw = fjord_width_avg[d]
    print(f'    FjORD width d={d:2d}: F1={fw["f1"]:.4f} +/- {fw["f1_std"]:.4f}')
print(f'  Edge confirm : F1={edge_confirmation_run["f1"]:.4f} (single seed, d={EDGE_TIER_DIMS[0]})')


## Save NB1 bundle

In [ ]:
nb1_bundle = {
    'central_avg':      central_avg,
    'central_runs':     central_runs,
    'localonly_avg':    localonly_avg,
    'localonly_runs':   localonly_runs,
    'fedavg_avg':       fedavg_avg,
    'fedavg_runs':      fedavg_runs,
    'fedsage_avg':      fedsage_avg,
    'fedsage_runs':     fedsage_runs,
    'fedsage_gat_avg':  fedsage_gat_avg,
    'fedsage_gat_runs': fedsage_gat_runs,
    'fedproto_avg':     fedproto_avg,
    'fedproto_runs':    fedproto_runs,
    # FIX A3: MOON baseline
    'moon_avg':         moon_avg,
    'moon_runs':        moon_runs,
    # FIX C1: SCAFFOLD baseline
    'scaffold_avg':     scaffold_avg,
    'scaffold_runs':    scaffold_runs,

    # EP-FedProto v3: new NB1 baselines + infra for NB2/NB3/NB4
    'device_tier_dims':       DEVICE_TIER_DIMS,
    'fixed_tier_avg':         fixed_tier_avg,
    'fixed_tier_runs':        fixed_tier_runs,
    'fjord_avg':              fjord_avg,
    'fjord_runs':              fjord_runs,
    'edge_cfg':                EDGE_CFG,
    'edge_tier_dims':          EDGE_TIER_DIMS,
    'edge_confirmation_run':   edge_confirmation_run,
    'scalability_n_clients':   SCALABILITY_N_CLIENTS,
    'scalability_splits':      scalability_splits,
}
save_ckpt('nb1_bundle', nb1_bundle)
print('nb1_bundle saved.')
print('Next: Save & Run All → add this notebook output as input to NB2.')
